In [ ]:
USR_NAME = 'thebeo2004'
REPO_NAME = 'AMBER'

!git clone https://{USR_NAME}@github.com/{USR_NAME}/{REPO_NAME}.git
%cd /kaggle/working/{REPO_NAME}
!git checkout cascade-anomaly
!git submodule update --init --recursive

Cloning into 'SAGE'...
remote: Enumerating objects: 881, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 881 (delta 60), reused 104 (delta 35), pack-reused 745 (from 1)
Receiving objects: 100% (881/881), 55.99 MiB | 34.73 MiB/s, done.
Resolving deltas: 100% (466/466), done.
/kaggle/working/SAGE
Branch 'cascade-anomaly' set up to track remote branch 'cascade-anomaly' from 'origin'.
Switched to a new branch 'cascade-anomaly'
Submodule 'anomalib' (https://github.com/thebeo2004/anomalib.git) registered for path 'anomalib'
Submodule 'detrex' (https://github.com/thebeo2004/detrex.git) registered for path 'detrex'
Cloning into '/kaggle/working/SAGE/anomalib'...
Cloning into '/kaggle/working/SAGE/detrex'...
Submodule path 'anomalib': checked out 'c681709a989701e72fee128ad80443be4d776b47'
Submodule path 'detrex': checked out '58a09c6fbe022077f6e26d8a8eda25f142092f21'
Submodule 'detectron2' (https://github.com/thebeo2004/dete

In [2]:
!uv pip install --system -q -r pyproject.toml
!pip install -q ninja
%cd detrex/detectron2
!uv pip install --system -q -e . --no-build-isolation
%cd ..
!uv pip install --system -q -e . --no-build-isolation
%cd /kaggle/working/{REPO_NAME}/anomalib
!uv pip install --system -q -e .

/kaggle/working/SAGE/detrex/detectron2
/kaggle/working/SAGE/detrex
/kaggle/working/SAGE/anomalib


In [ ]:
%%writefile /kaggle/working/run_inference.py

import os
import sys
import numpy as np

os.chdir('/kaggle/working/AMBER')

# 2. Đưa SAGE vào sys.path để đảm bảo import tuyệt đối chính xác
if '/kaggle/working/AMBER' not in sys.path:
    sys.path.insert(0, '/kaggle/working/AMBER')

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
from src.memory_bank.anomaly_detection import init_anomaly_model_for_inference, run_inference
from src.memory_bank.binary_cxr_dataset import BinaryCXRDataset
from src.memory_bank.binary_classification import binary_classification_inference

raddino_checkpoint_path = "/kaggle/input/models/thebeo182004/raddino-vitb14-topad16/pytorch/default/1/rad_dino_vitb14_detrex_ready.pth"

test_normal_dir = "/kaggle/input/datasets/thebeo182004/medanomaly-vindrcxr/medi_anomaly/test/normal"
test_abnormal_dir = "/kaggle/input/datasets/thebeo182004/medanomaly-vindrcxr/medi_anomaly/test/abnormal"

test_normal_imgs = [
    os.path.join(test_normal_dir, img) for img in os.listdir(test_normal_dir)
]

test_abnormal_imgs = [
    os.path.join(test_abnormal_dir, img) for img in os.listdir(test_abnormal_dir)
]

print(f"Total Normal Images: {len(test_normal_imgs)}, Total Abnormal Images: {len(test_abnormal_imgs)}")

memory_bank_dirs = [
    '/kaggle/input/notebooks/thebeo182004/1-1-1-anomalyraddino-50-1-0',
    '/kaggle/input/notebooks/thebeo2004/1-1-2-anomalyraddino-100-0-5',
    '/kaggle/input/notebooks/thebeo182004/1-1-3-anomalyraddino-250-0-2',
    '/kaggle/input/notebooks/chihoangf/1-1-4-anomalyraddino-500-0-1',
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov1-vindrcxr-1-1-0-1-8",
    "/kaggle/input/notebooks/thebeo2004/anomalyraddinov1-vindrcxr-5-1-0-1-9",
    "/kaggle/input/notebooks/thebeo2004/anomalyraddinov1-vindrcxr-10-1-0-1-10",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov1-vindrcxr-20-1-0-1-11",
]

cases = [
    '1', '2', '3', '4', '8', '9', '10', '11'
]

memory_bank_path = 'raddino_memory_bank.pt'

save_dir = '/kaggle/working/inference'
os.makedirs(save_dir, exist_ok=True)

import numpy as np

def save_inference_results(save_path: str, true_labels: np.ndarray, predicted_scores_per_k: dict):

    data_to_save = {
        'true_labels': true_labels
    }

    for k, strats in predicted_scores_per_k.items():
        for strategy_name, scores in strats.items():
            key = f"k_{k}_{strategy_name}"
            data_to_save[key] = scores 
        
    np.savez(save_path, **data_to_save)
    print(f"Successfully save at: {save_path}")

for case, mem_bank in zip(cases, memory_bank_dirs):
    
    # save_dir = os.path.join('/kaggle/working/inference', case)
    
    memory_bank_path = os.path.join(mem_bank, 'raddino_memory_bank.pt')

    true_labels, anomaly_scores = binary_classification_inference(
        raddino_checkpoint_path=raddino_checkpoint_path,
        memory_bank_path=memory_bank_path,
        normal_paths=test_normal_imgs,
        abnormal_paths=test_abnormal_imgs,
        batch_size=1,
        num_neighbours=[1, 2, 4, 8, 10, 15, 20]
    )

    save_inference_results(
        os.path.join(save_dir, f"inference_case_{case}.npz"),
        true_labels,
        anomaly_scores
    )


Writing /kaggle/working/run_inference.py


In [4]:
!python /kaggle/working/run_inference.py

/usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:66: SyntaxWarning: invalid escape sequence '\m'
  Initial value for the global affine scaling :math:`s_\mathrm{global}`.
/kaggle/working/SAGE/detrex/detrex/layers/dcn_v3.py:23: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/SAGE/detrex/detrex/layers/dcn_v3.py:52: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Total Normal Images: 1000, Total Abnormal Images: 1000
[06/15 15:07:08 timm backbone]: backbone out_indices: (11